# LlamaIndex: assistente para consultar um banco de dados SQL
Conexão a um banco SQLite com SQLAlchemy, indexação das tabelas com embeddings para que o LlamaIndex encontre a
tabela relevante para cada pergunta, geração de consultas SQL com uma LLM da Groq, prompts personalizados,
um pipeline com Workflow e um chat em Gradio.

**Origem:** Alura — curso de LlamaIndex com bancos de dados SQL (dados da loja Zoop), com adaptações próprias.

**Dados:** arquivo `ecommerce.db` (SQLite), a ser enviado para `/content/` no Colab.

**Secrets do Colab:** `GROQ_API` (chave da API da Groq).

**Índice**
1. Conectando ao banco de dados
2. Configurando o LlamaIndex: LLM e embeddings
3. Indexando as tabelas
4. Consultando o banco com SQLTableRetrieverQueryEngine
5. Contextualizando o modelo com descrições das tabelas
6. Personalizando os prompts
7. Construindo a cadeia de ações com Workflow
8. Chat com o assistente SQL (Gradio)

In [ ]:
!pip install -q -U llama-index llama-index-llms-groq llama-index-embeddings-huggingface gradio

from typing import List

import gradio as gr
import pandas as pd
from google.colab import userdata
from llama_index.core import PromptTemplate, Settings, SQLDatabase, VectorStoreIndex
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine
from llama_index.core.llms import ChatMessage, ChatResponse
from llama_index.core.objects import ObjectIndex, SQLTableNodeMapping, SQLTableSchema
from llama_index.core.prompts.default_prompts import DEFAULT_TEXT_TO_SQL_PROMPT
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from sqlalchemy import MetaData, create_engine

## 1. Conectando ao banco de dados
Usamos a biblioteca [SQLAlchemy](https://docs.sqlalchemy.org/en/20/) para trabalhar com bancos de dados em Python:
- [`create_engine`](https://docs.sqlalchemy.org/en/20/core/engines.html) cria o ponto central de conexão com o
  banco a partir da URL, gerenciando as conexões e a execução de operações SQL;
- [`MetaData`](https://docs.sqlalchemy.org/en/20/core/metadata.html) é a coleção de informações sobre o esquema
  (tabelas e colunas); `reflect` examina o banco conectado e preenche esse objeto.

In [ ]:
url = '/content/ecommerce.db'
engine = create_engine(f'sqlite:///{url}')

metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)

# Dicionário cujas chaves são os nomes das tabelas disponíveis
metadata_obj.tables.keys()

[`pd.read_sql_table`](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_table.html) carrega uma tabela
diretamente para um DataFrame.

In [ ]:
pd.read_sql_table('Clientes', engine).head()

## 2. Configurando o LlamaIndex: LLM e embeddings
Modelos utilizados:
- LLM [`openai/gpt-oss-20b`](https://console.groq.com/docs/models) servida pela Groq
  ([guia Groq no LlamaIndex](https://docs.llamaindex.ai/en/stable/examples/llm/groq/));
- embeddings [`BAAI/bge-m3`](https://huggingface.co/BAAI/bge-m3) (Hugging Face, multilíngue).

**Embeddings** são representações numéricas de dados (palavras, frases, imagens...) em um espaço
multidimensional. Eles transformam dados complexos em vetores que capturam significados e padrões, de modo que
itens com significados semelhantes fiquem próximos nesse espaço. São gerados por modelos de aprendizado de
máquina treinados para extrair relações nos dados — em embeddings de palavras, por exemplo, o modelo aprende
associações a partir do contexto em que as palavras aparecem. Aqui eles servem para encontrar a tabela mais
relacionada à pergunta do usuário.

In [ ]:
key = userdata.get('GROQ_API')

modelo = 'openai/gpt-oss-20b'
modelo_hf_emb = 'BAAI/bge-m3'

llm = Groq(model=modelo, api_key=key)

Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(model_name=modelo_hf_emb)

## 3. Indexando as tabelas
- `SQLDatabase` conecta o banco de dados ao LlamaIndex;
- `SQLTableNodeMapping` mapeia as tabelas do banco para os nós do índice;
- `SQLTableSchema` representa cada tabela como um objeto (nome e, opcionalmente, uma descrição);
- `ObjectIndex` combina os esquemas e o mapeamento para construir um índice vetorial com embeddings;
- o `retriever` encontra a tabela mais relevante para a pergunta (`similarity_top_k=1`).

In [ ]:
sql_database = SQLDatabase(engine)
table_node_mapping = SQLTableNodeMapping(sql_database)

table_schema_objs = [SQLTableSchema(table_name=nome_tabela) for nome_tabela in metadata_obj.tables.keys()]

obj_index = ObjectIndex.from_objects(table_schema_objs, table_node_mapping, VectorStoreIndex)
obj_retriever = obj_index.as_retriever(similarity_top_k=1)

## 4. Consultando o banco com SQLTableRetrieverQueryEngine
O `SQLTableRetrieverQueryEngine` combina o banco e o índice: recupera a tabela relevante, gera a consulta SQL a
partir da pergunta em linguagem natural, executa e sintetiza a resposta. A consulta SQL gerada fica em
`resposta.metadata`.

In [ ]:
query_engine = SQLTableRetrieverQueryEngine(sql_database, obj_retriever)

resposta = query_engine.query('Quais estados mais frequentes na tabela Clientes?')
print(resposta)
resposta.metadata

In [ ]:
resposta = query_engine.query('Quais as fornecedoras localizadas na cidade de São Paulo?')
print(resposta)

In [ ]:
resposta = query_engine.query('Quantos funcionários com o cargo de vendedores temos cadastrados?')
print(resposta)

## 5. Contextualizando o modelo com descrições das tabelas
Só com o nome das tabelas o modelo pode escolher a tabela errada ou errar valores de filtro. Pedimos à LLM uma
descrição curta de cada tabela a partir de uma amostra dos dados; essas descrições entram no esquema das
tabelas (`context_str`) e o índice é reconstruído.

In [ ]:
# Gera a descrição de uma tabela com base em uma amostra dos dados
def gerar_descricao_tabela(nome_tabela, df_amostra):
    prompt = f"""
    Analise a amostra da tabela '{nome_tabela}' abaixo e forneça uma curta e breve descrição do conteúdo dessa tabela.
    Informe até o máximo de 5 valores únicos de cada coluna.


    Amostra da Tabela:
    {df_amostra}

    Descrição:
    """
    resposta = llm.complete(prompt=prompt)
    return resposta.text

In [ ]:
nomes_tabelas = metadata_obj.tables.keys()
dicionario_tabelas = {}

for nome_tabela in nomes_tabelas:
    df = pd.read_sql_table(nome_tabela, engine)
    df_amostra = df.head(5).to_string()

    descricao = gerar_descricao_tabela(nome_tabela, df_amostra)
    dicionario_tabelas[nome_tabela] = descricao
    print(f'Tabela: {nome_tabela}\n Descrição: {descricao}')
    print('-' * 15)

In [ ]:
# A descrição de cada tabela precisa ser adicionada ao seu schema antes de recriar o índice
table_schema_objs = [
    SQLTableSchema(table_name=nome_tabela, context_str=dicionario_tabelas[nome_tabela])
    for nome_tabela in nomes_tabelas
]

obj_index = ObjectIndex.from_objects(table_schema_objs, table_node_mapping, VectorStoreIndex)
obj_retriever = obj_index.as_retriever(similarity_top_k=1)

query_engine_2 = SQLTableRetrieverQueryEngine(sql_database, obj_retriever)

In [ ]:
resposta = query_engine_2.query('Quais as fornecedoras localizadas na cidade de São Paulo?')
print(resposta)

In [ ]:
resposta = query_engine_2.query('Quantos funcionários com o cargo de vendedores temos cadastrados?')
print(resposta)

## 6. Personalizando os prompts
### 6.1 Prompt 1: texto para SQL
Partimos do prompt padrão do LlamaIndex (`DEFAULT_TEXT_TO_SQL_PROMPT`) e criamos uma versão em português.
`PromptTemplate` cria um modelo de prompt com variáveis (`{dialect}`, `{schema}`, `{pergunta_user}`);
o dialeto do banco já é preenchido na criação.

In [ ]:
print(DEFAULT_TEXT_TO_SQL_PROMPT.template)

In [ ]:
texto2sql = """Dada uma pergunta em linguagem natural, crie uma consulta {dialect} sintaticamente correta para executar e, em seguida, verifique os resultados da consulta e retorne a resposta. Você pode ordenar os resultados por uma coluna relevante para retornar os exemplos mais informativos no banco de dados.

Nunca consulte todas as colunas de uma tabela específica. Pergunte apenas por algumas colunas relevantes, de acordo com a pergunta.

Preste atenção para usar apenas os nomes de colunas que você pode ver na descrição do esquema. Tenha cuidado para não consultar colunas que não existem. Preste atenção em qual coluna está em qual tabela. Além disso, qualifique os nomes das colunas com o nome da tabela quando necessário.

Use o seguinte formato, cada um em uma linha:

Pergunta: Pergunta aqui
ConsultaSQL: Consulta SQL para executar
ResultadoSQL: Resultado da ConsultaSQL
Resposta: Resposta final aqui

Use apenas as tabelas listadas abaixo.

{schema}

Pergunta: {pergunta_user}
ConsultaSQL:
"""

prompt_1 = PromptTemplate(texto2sql, dialect=engine.dialect.name)
print(prompt_1.template)

### 6.2 Descrição das tabelas, consulta SQL e resultado
- `descricao_tabela` monta o texto do `{schema}`: estrutura de cada tabela recuperada + a descrição gerada na seção 5;
- `resposta_sql` extrai da resposta da LLM apenas a consulta (o trecho entre `ConsultaSQL:` e `ResultadoSQL:`);
- `SQLRetriever` executa a consulta SQL diretamente no banco configurado.

In [ ]:
def descricao_tabela(schema_tabelas: List[SQLTableSchema]):
    descricao_str = []
    for tabela_schema in schema_tabelas:
        info_tabela = sql_database.get_single_table_info(tabela_schema.table_name)
        context_str_value = tabela_schema.context_str if tabela_schema.context_str is not None else ''
        info_tabela += (' A descrição da tabela é: ' + context_str_value)
        descricao_str.append(info_tabela)
    return '\n\n'.join(descricao_str)


def resposta_sql(resposta: ChatResponse) -> str:
    conteudo_resposta = resposta.message.content
    sql_consulta = conteudo_resposta.split('ConsultaSQL: ', 1)[-1].split('ResultadoSQL: ', 1)[0]
    return sql_consulta.strip().strip('```').strip()


resultado_sql = SQLRetriever(sql_database)

### 6.3 Prompt 2: resposta em linguagem natural
Com a pergunta, a consulta SQL e o resultado, a LLM elabora a resposta final de forma objetiva.

In [ ]:
prompt_2_str = '''
    Você é o "Assistente mais habilidoso em consulta de banco de dados".
    Dada a seguinte pergunta, a consulta SQL correspondente e o resultado SQL, responda à pergunta de modo agradável e objetiva.
    Evite iniciar conversas com cumprimentos e apresentações, seja profissional.

    Pergunta: {pergunta_user}
    Consulta SQL: {consulta}
    Resultado SQL: {resultado}
    Resposta:
    '''

prompt_2 = PromptTemplate(prompt_2_str)

## 7. Construindo a cadeia de ações com Workflow
A cadeia completa é: pergunta → tabela relevante (`obj_retriever`) → descrição das tabelas → prompt 1 → LLM →
extração da consulta SQL → execução no banco (`SQLRetriever`) → prompt 2 → LLM → resposta.

No curso essa cadeia era montada com `QueryPipeline` (`add_chain`/`add_link` e `FnComponent`), removido nas versões
recentes do LlamaIndex. Aqui cada etapa é um `@step` de um **Workflow**, e os eventos carregam os dados entre elas.

In [ ]:
class SchemaEvent(Event):
    pergunta_user: str
    schema: str


class ConsultaEvent(Event):
    pergunta_user: str
    consulta: str


class ResultadoEvent(Event):
    pergunta_user: str
    consulta: str
    resultado: str


class SQLWorkflow(Workflow):

    # entrada -> acesso_tabela -> contexto_tabela
    @step
    async def recuperar_tabelas(self, ev: StartEvent) -> SchemaEvent:
        tabelas = obj_retriever.retrieve(ev.query)
        return SchemaEvent(pergunta_user=ev.query, schema=descricao_tabela(tabelas))

    # prompt_1 -> llm_1 -> consulta_sql
    @step
    async def gerar_consulta(self, ev: SchemaEvent) -> ConsultaEvent:
        prompt = prompt_1.format(schema=ev.schema, pergunta_user=ev.pergunta_user)
        resposta = await llm.achat([ChatMessage(role='user', content=prompt)])
        return ConsultaEvent(pergunta_user=ev.pergunta_user, consulta=resposta_sql(resposta))

    # resultado_sql
    @step
    async def executar_consulta(self, ev: ConsultaEvent) -> ResultadoEvent:
        nos = resultado_sql.retrieve(ev.consulta)
        resultado = '\n'.join(no.get_content() for no in nos)
        return ResultadoEvent(pergunta_user=ev.pergunta_user, consulta=ev.consulta, resultado=resultado)

    # prompt_2 -> llm_2
    @step
    async def responder(self, ev: ResultadoEvent) -> StopEvent:
        prompt = prompt_2.format(pergunta_user=ev.pergunta_user, consulta=ev.consulta, resultado=ev.resultado)
        resposta = await llm.acomplete(prompt)
        return StopEvent(result=str(resposta))


qp = SQLWorkflow(timeout=120, verbose=False)

In [ ]:
resposta = await qp.run(query='Quais estados mais frequentes na tabela Clientes?')
print(resposta)

In [ ]:
resposta = await qp.run(query='Quantos funcionários com o cargo de vendedores temos cadastrados?')
print(resposta)

## 8. Chat com o assistente SQL (Gradio)
- `entrada_saida` executa a cadeia para a mensagem do usuário;
- `atualizar_historico` adiciona pergunta e resposta ao histórico do chat (formato de mensagens `role`/`content`);
- a interface tem o chat, a caixa de texto (Enter envia) e um botão para limpar a conversa.

In [ ]:
async def entrada_saida(msg_user: str):
    return str(await qp.run(query=msg_user))


async def atualizar_historico(msg_user, historico):
    historico = historico or []
    msg_assistente = await entrada_saida(msg_user)
    historico.append({'role': 'user', 'content': msg_user})
    historico.append({'role': 'assistant', 'content': msg_assistente})
    return '', historico

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('## Chat com Assistente SQL')
    gr.Markdown(
        '''
        Este é um assistente SQL interativo, projetado para responder perguntas sobre os dados da loja Zoop.
        Insira sua pergunta no campo abaixo e o assistente irá responder com base no resultado da consulta SQL
        nos dados disponíveis.
        '''
    )
    chatbot = gr.Chatbot(label='Chat com Assistente')
    msg = gr.Textbox(label='Digite a sua pergunta e tecle Enter para enviar',
                     placeholder='Digite o texto aqui.')
    limpeza = gr.Button('Limpar a conversa')

    msg.submit(atualizar_historico, inputs=[msg, chatbot], outputs=[msg, chatbot], queue=False)
    limpeza.click(lambda: [], inputs=None, outputs=chatbot, queue=False)

demo.queue()
demo.launch(debug=False, share=True)